# Extração CNES — Cadastro Nacional de Estabelecimentos de Saúde para o projeto Conecta Saúde

Este notebook documenta a fonte **CNES — Cadastro Nacional de Estabelecimentos de Saúde**.

O objetivo é mostrar:

1. **quais dados serão extraídos**;
2. **qual endpoint público será usado**;
3. **como realizar a extração com Python**;
4. **quais campos são úteis para o projeto**;
5. **quais perguntas de negócio podem ser respondidas com esses dados**.

> Observação importante: o CNES não é consumido, para análise massiva, como uma API REST transacional. A forma pública mais adequada para pipeline analítico é o download dos arquivos mensais em formato **DBC** no servidor público do DATASUS. Neste notebook, esse servidor será tratado como **endpoint público de arquivos**. O pacote `pysus` também será usado como camada de conveniência para baixar e ler os dados.


## 1. Papel do CNES no projeto

Dentro do **Painel Inteligente de Pressão Assistencial e Priorização Hospitalar**, o CNES será a fonte principal para medir **capacidade instalada** e características dos estabelecimentos de saúde.

A base será usada para analisar:

- hospitais e unidades de saúde existentes por município;
- tipo de estabelecimento;
- natureza jurídica e gestão;
- quantidade de leitos existentes e leitos SUS;
- serviços especializados disponíveis;
- habilitações;
- profissionais vinculados ao estabelecimento;
- equipamentos disponíveis;
- vínculo entre o código CNES do SIH/SUS e os atributos reais da unidade.

Na arquitetura do projeto, o CNES será cruzado com:

- **SIH/SUS**, para calcular internações por leito, permanência média por hospital e ranking de hospitais sob pressão;
- **IBGE**, para calcular leitos por 10 mil habitantes e capacidade proporcional por município;
- **SIA/SUS**, para analisar demanda ambulatorial versus oferta de serviços e especialidades.


## 2. Endpoint público de conexão

A estrutura dos arquivos CNES no servidor público do DATASUS segue o padrão:

```text
https://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/{GRUPO}/{GRUPO}{UF}{AAMM}.dbc
```

Exemplo para estabelecimentos de São Paulo, janeiro de 2024:

```text
https://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/ST/STSP2401.dbc
```

Exemplo para leitos de São Paulo, janeiro de 2024:

```text
https://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/LT/LTSP2401.dbc
```

Grupos mais úteis para o projeto:

| Grupo | Conteúdo | Uso no projeto |
|---|---|---|
| `ST` | Estabelecimentos | Nome, município, tipo, gestão e características gerais da unidade |
| `LT` | Leitos | Leitos existentes, contratados e SUS por estabelecimento |
| `PF` | Profissionais | Profissionais por ocupação/CBO e estabelecimento |
| `EQ` | Equipamentos | Equipamentos disponíveis por unidade |
| `SR` | Serviços especializados | Serviços ofertados por estabelecimento |
| `HB` | Habilitações | Habilitações e capacidade técnica da unidade |

Para o MVP, recomendo começar por `ST` e `LT`, pois são suficientes para calcular os principais indicadores de capacidade hospitalar.


In [ ]:
# Instalação das dependências
# Execute esta célula no Colab/Jupyter caso os pacotes ainda não estejam instalados.

%pip install -q pandas pyarrow requests pysus


## 3. Parâmetros iniciais da extração

Para um primeiro teste, recomenda-se usar uma única UF e um único mês. Depois, o pipeline pode ser ampliado para várias UFs, meses e grupos.


In [ ]:
from pathlib import Path
from typing import Iterable
import pandas as pd
import requests

# Parâmetros de teste
UF = "SP"       # Ex.: "SP", "RJ", "MG", "AC"
ANO = 2024
MESES = [1]     # Ex.: [1, 2, 3]
GRUPOS = ["ST", "LT"]  # ST = estabelecimentos; LT = leitos

BASE_URL_CNES = "https://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/"
DIRETORIO_RAW = Path("dados/raw/cnes")
DIRETORIO_TRATADO = Path("dados/tratado/cnes")

DIRETORIO_RAW.mkdir(parents=True, exist_ok=True)
DIRETORIO_TRATADO.mkdir(parents=True, exist_ok=True)


## 4. Geração do link do arquivo DBC

A função abaixo monta automaticamente a URL pública para qualquer grupo CNES, UF, ano e mês.


In [ ]:
def montar_url_cnes(grupo: str, uf: str, ano: int, mes: int) -> str:
    '''Monta a URL pública de um arquivo CNES no servidor DATASUS.'''
    grupo = grupo.upper().strip()
    uf = uf.upper().strip()
    aa = str(ano)[-2:]
    mm = str(mes).zfill(2)
    arquivo = f"{grupo}{uf}{aa}{mm}.dbc"
    return f"{BASE_URL_CNES}{grupo}/{arquivo}"

for grupo in GRUPOS:
    for mes in MESES:
        print(montar_url_cnes(grupo, UF, ANO, mes))


## 5. Verificação de disponibilidade

Alguns grupos podem não existir para todos os meses/UFs, ou podem ter atraso de publicação. Por isso, validamos o status HTTP antes da extração.


In [ ]:
def verificar_disponibilidade(url: str, timeout: int = 30) -> dict:
    try:
        resposta = requests.head(url, timeout=timeout, allow_redirects=True)
        return {
            "url": url,
            "status_code": resposta.status_code,
            "disponivel": resposta.status_code == 200,
            "content_length_bytes": resposta.headers.get("Content-Length"),
            "content_type": resposta.headers.get("Content-Type"),
        }
    except requests.RequestException as erro:
        return {
            "url": url,
            "status_code": None,
            "disponivel": False,
            "erro": str(erro),
        }

checks = []
for grupo in GRUPOS:
    for mes in MESES:
        checks.append(verificar_disponibilidade(montar_url_cnes(grupo, UF, ANO, mes)))

df_disponibilidade = pd.DataFrame(checks)
df_disponibilidade


## 6. Extração com PySUS

O `pysus` possui função simplificada para CNES:

```python
from pysus.api import cnes
df = cnes(state="SP", year=2024, month=1, group="ST")
```

A função abaixo encapsula essa chamada e tenta manter compatibilidade com versões em que `month` aceita lista ou valor único.


In [ ]:
def extrair_cnes_pysus(uf: str, ano: int, meses: Iterable[int], grupos: Iterable[str]) -> dict:
    '''
    Extrai grupos do CNES usando a API simplificada do PySUS.

    Retorna um dicionário no formato {grupo: DataFrame}.
    '''
    from pysus.api import cnes

    resultados = {}
    uf = uf.upper().strip()

    for grupo in grupos:
        grupo = grupo.upper().strip()
        frames = []
        for mes in meses:
            try:
                df = cnes(state=uf, year=ano, month=mes, group=grupo)
            except TypeError:
                df = cnes(state=uf, year=ano, month=mes)

            if not isinstance(df, pd.DataFrame):
                try:
                    df = df.to_pandas()
                except AttributeError:
                    df = pd.DataFrame(df)

            df["FONTE_GRUPO_CNES"] = grupo
            df["UF_PARAMETRO"] = uf
            df["ANO_PARAMETRO"] = ano
            df["MES_PARAMETRO"] = mes
            frames.append(df)

        resultados[grupo] = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    return resultados

bases_cnes = extrair_cnes_pysus(UF, ANO, MESES, GRUPOS)

for grupo, df in bases_cnes.items():
    print(f"Grupo {grupo}: {df.shape[0]:,} linhas e {df.shape[1]:,} colunas")
    display(df.head())


## 7. Quais dados serão extraídos

Os layouts do CNES podem variar por grupo e competência. Para o projeto, os dados de maior valor são:

### Grupo `ST` — estabelecimentos

| Campo esperado | Uso no projeto |
|---|---|
| Código CNES | Chave de integração com SIH/SUS e SIA/SUS |
| Município do estabelecimento | Localização da capacidade instalada |
| Nome fantasia / razão social | Identificação da unidade no dashboard |
| Tipo de unidade | Classificação da unidade: hospital, UBS, policlínica etc. |
| Natureza jurídica | Público, privado, filantrópico ou outro vínculo |
| Gestão | Municipal, estadual, dupla ou federal |
| Atividade / nível de atenção | Apoio à classificação assistencial |

### Grupo `LT` — leitos

| Campo esperado | Uso no projeto |
|---|---|
| Código CNES | Vincular leitos ao estabelecimento |
| Tipo de leito | Clínico, cirúrgico, UTI, obstétrico etc. |
| Quantidade existente | Capacidade física total |
| Quantidade SUS | Capacidade disponível ao SUS |
| Quantidade contratada | Oferta contratual da unidade |

### Grupos complementares

| Grupo | Uso |
|---|---|
| `PF` | Quantidade de profissionais e CBO por unidade |
| `EQ` | Equipamentos disponíveis, como tomógrafos, raio-x, respiradores etc. |
| `SR` | Serviços especializados ofertados |
| `HB` | Habilitações técnicas da unidade |


In [ ]:
# Visualizar colunas disponíveis em cada grupo extraído
for grupo, df in bases_cnes.items():
    print(f"\n--- Colunas do grupo {grupo} ---")
    display(pd.DataFrame({
        "coluna": df.columns,
        "tipo_detectado": [str(df[col].dtype) for col in df.columns]
    }))


## 8. Seleção dinâmica de campos úteis

Como os nomes podem variar ligeiramente entre versões, a seleção abaixo usa uma lista de candidatos e mantém apenas o que existir no arquivo extraído.


In [ ]:
CANDIDATAS_ST = [
    "CNES", "CODUFMUN", "COD_CEP", "CPF_CNPJ", "PF_PJ", "RAZAO_SOC", "NOME_FANT",
    "TP_UNID", "NIV_DEP", "CNPJ_MAN", "NAT_JUR", "GESTAO", "ATIVIDAD", "RETENCAO",
    "MUNICIPIO", "UF", "FONTE_GRUPO_CNES", "UF_PARAMETRO", "ANO_PARAMETRO", "MES_PARAMETRO"
]

CANDIDATAS_LT = [
    "CNES", "CODUFMUN", "TP_LEITO", "CODLEITO", "QT_EXIST", "QT_CONTR", "QT_SUS",
    "COMPETEN", "FONTE_GRUPO_CNES", "UF_PARAMETRO", "ANO_PARAMETRO", "MES_PARAMETRO"
]

def selecionar_colunas(df: pd.DataFrame, candidatas: list[str]) -> pd.DataFrame:
    colunas = [c for c in candidatas if c in df.columns]
    return df[colunas].copy()

cnes_st = selecionar_colunas(bases_cnes.get("ST", pd.DataFrame()), CANDIDATAS_ST)
cnes_lt = selecionar_colunas(bases_cnes.get("LT", pd.DataFrame()), CANDIDATAS_LT)

print("ST selecionado:", cnes_st.shape)
display(cnes_st.head())

print("LT selecionado:", cnes_lt.shape)
display(cnes_lt.head())


## 9. Tratamento básico dos tipos de dados

Os campos de quantidade de leitos precisam ser convertidos para numérico para permitir agregações.


In [ ]:
def tratar_quantidades_leitos(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ["QT_EXIST", "QT_CONTR", "QT_SUS"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
    return df

cnes_lt = tratar_quantidades_leitos(cnes_lt)
cnes_lt.info()


## 10. Dicionário analítico dos campos selecionados


In [ ]:
DICIONARIO_CNES = pd.DataFrame([
    {"campo": "CNES", "descricao": "Código nacional do estabelecimento; chave para cruzar com SIH/SUS e SIA/SUS."},
    {"campo": "CODUFMUN", "descricao": "Código do município do estabelecimento."},
    {"campo": "RAZAO_SOC", "descricao": "Razão social da unidade."},
    {"campo": "NOME_FANT", "descricao": "Nome fantasia apresentado ao usuário final."},
    {"campo": "TP_UNID", "descricao": "Tipo de unidade de saúde."},
    {"campo": "NAT_JUR", "descricao": "Natureza jurídica do estabelecimento."},
    {"campo": "GESTAO", "descricao": "Tipo de gestão da unidade."},
    {"campo": "TP_LEITO/CODLEITO", "descricao": "Classificação do leito."},
    {"campo": "QT_EXIST", "descricao": "Quantidade de leitos existentes."},
    {"campo": "QT_SUS", "descricao": "Quantidade de leitos disponíveis ao SUS."},
])

DICIONARIO_CNES


## 11. Salvamento das bases tratadas


In [ ]:
if not cnes_st.empty:
    arq_st = DIRETORIO_TRATADO / f"cnes_st_{UF}_{ANO}_{'-'.join(str(m).zfill(2) for m in MESES)}.parquet"
    cnes_st.to_parquet(arq_st, index=False)
    print(f"Arquivo ST salvo em: {arq_st}")

if not cnes_lt.empty:
    arq_lt = DIRETORIO_TRATADO / f"cnes_lt_{UF}_{ANO}_{'-'.join(str(m).zfill(2) for m in MESES)}.parquet"
    cnes_lt.to_parquet(arq_lt, index=False)
    print(f"Arquivo LT salvo em: {arq_lt}")


## 12. Perguntas que conseguimos responder com CNES

Com a base extraída, conseguimos responder perguntas ligadas à **capacidade instalada**:

1. Quantos estabelecimentos de saúde existem por município?
2. Quais municípios possuem hospitais e quais possuem apenas unidades básicas ou ambulatoriais?
3. Quantos leitos existem por hospital?
4. Quantos leitos SUS existem por município?
5. Quais hospitais concentram maior oferta de leitos?
6. Quais municípios têm baixa capacidade instalada em relação à demanda observada no SIH/SUS?
7. Quais unidades possuem serviços especializados?
8. Quais estabelecimentos possuem equipamentos ou habilitações relevantes?
9. Quais regiões têm maior fragilidade estrutural?
10. Quais hospitais devem entrar no ranking de pressão assistencial?

Quando cruzado com SIH/SUS e IBGE:

- **internações por leito SUS**;
- **leitos por 10 mil habitantes**;
- **ranking de hospitais sob pressão**;
- **ranking de municípios com alta demanda e baixa capacidade**;
- **identificação de vazios assistenciais**.


## 13. Exemplo 1 — Leitos por estabelecimento


In [ ]:
if not cnes_lt.empty and "CNES" in cnes_lt.columns:
    agg_dict = {}
    if "QT_EXIST" in cnes_lt.columns:
        agg_dict["leitos_existentes"] = ("QT_EXIST", "sum")
    if "QT_SUS" in cnes_lt.columns:
        agg_dict["leitos_sus"] = ("QT_SUS", "sum")
    if "QT_CONTR" in cnes_lt.columns:
        agg_dict["leitos_contratados"] = ("QT_CONTR", "sum")

    leitos_por_cnes = cnes_lt.groupby("CNES").agg(**agg_dict).reset_index()
    display(leitos_por_cnes.sort_values(list(agg_dict.keys())[0], ascending=False).head(20))
else:
    print("Base de leitos não carregada ou sem coluna CNES.")


## 14. Exemplo 2 — Estabelecimentos por município


In [ ]:
if not cnes_st.empty:
    coluna_municipio = "CODUFMUN" if "CODUFMUN" in cnes_st.columns else None
    if coluna_municipio:
        estabelecimentos_municipio = (
            cnes_st.groupby(coluna_municipio)
            .agg(qtd_estabelecimentos=("CNES", "nunique") if "CNES" in cnes_st.columns else (coluna_municipio, "size"))
            .reset_index()
            .sort_values("qtd_estabelecimentos", ascending=False)
        )
        display(estabelecimentos_municipio.head(20))
    else:
        print("Coluna de município não encontrada.")
else:
    print("Base ST vazia.")


## 15. Como o CNES entra no índice de pressão assistencial

O CNES fornece o denominador da pressão assistencial:

```text
Internações por leito SUS = internações SIH/SUS / leitos SUS CNES
```

Também permite calcular:

```text
Leitos SUS por 10 mil habitantes = (leitos SUS CNES / população IBGE) * 10.000
```

Essas métricas ajudam a separar municípios que têm volume alto porque são grandes daqueles que realmente apresentam pressão estrutural acima da capacidade disponível.
